In [2]:
# התקנת הספריות הנדרשות
!pip install -q transformers datasets evaluate accelerate scikit-learn kaggle

import os
import shutil
import torch
import evaluate
import numpy as np
from datasets import load_dataset
from transformers import AutoImageProcessor, ViTForImageClassification, TrainingArguments, Trainer
from google.colab import userdata
from huggingface_hub import login, HfApi

# משיכת מפתחות הגישה מתוך הסודות של קולאב
os.environ['KAGGLE_USERNAME'] = userdata.get('KAGGLE_USERNAME')
os.environ['KAGGLE_KEY'] = userdata.get('KAGGLE_KEY')

print("מוריד את דטאסט FreshCheck הרשמי מקאגל...")
!kaggle datasets download -d orofinadedamola/freshcheck-fruit-ripeness-classification-dataset

print("מחלץ את הנתונים...")
!unzip -q -o freshcheck-fruit-ripeness-classification-dataset.zip -d freshcheck_data/

print("סורק את התיקיות ומסדר את תמונות העגבנייה...")
source_dir = "freshcheck_data"
target_dir = "clean_tomato_dataset"

if os.path.exists(target_dir):
    shutil.rmtree(target_dir)

image_count = 0
for root, dirs, files in os.walk(source_dir):
    for file in files:
        if file.lower().endswith(('.png', '.jpg', '.jpeg')):
            full_path = root.lower()

            # מוודאים שמדובר בעגבנייה
            if 'tomato' in full_path or 'tomato' in file.lower():
                class_name = None

                # סדר הבדיקה חשוב כדי לא להתבלבל בין המילים השונות
                if 'overripe' in full_path:
                    class_name = 'overripe'
                elif 'unripe' in full_path:
                    class_name = 'unripe'
                elif 'rotten' in full_path or 'damaged' in full_path:
                    class_name = 'damaged'
                elif 'ripe' in full_path:
                    class_name = 'ripe'

                if class_name:
                    class_dir = os.path.join(target_dir, class_name)
                    os.makedirs(class_dir, exist_ok=True)
                    # יצירת שם ייחודי כדי למנוע דריסת קבצים
                    new_file_name = f"{image_count}_{file}"
                    shutil.copy(os.path.join(root, file), os.path.join(class_dir, new_file_name))
                    image_count += 1

print(f"נמצאו וסודרו בהצלחה {image_count} תמונות.")

print("טוען את התמונות למבנה נתונים...")
dataset = load_dataset("imagefolder", data_dir=target_dir)

# חלוקה מחדש לאימון, ולידציה ובדיקה
splits = dataset['train'].train_test_split(test_size=0.2, seed=42)
train_ds = splits['train']
val_test_splits = splits['test'].train_test_split(test_size=0.5, seed=42)
val_ds = val_test_splits['train']
test_ds = val_test_splits['test']

label_names = train_ds.features['label'].names
id2label = {i: name for i, name in enumerate(label_names)}
label2id = {name: i for i, name in enumerate(label_names)}

print(f"המחלקות שזוהו בהצלחה: {label_names}")

# עיבוד מקדים לתמונות
model_name = "google/vit-base-patch16-224"
processor = AutoImageProcessor.from_pretrained(model_name)

def transforms(examples):
    inputs = processor([img.convert("RGB") for img in examples["image"]], return_tensors="pt")
    inputs["labels"] = examples["label"]
    return inputs

train_ds.set_transform(transforms)
val_ds.set_transform(transforms)
test_ds.set_transform(transforms)

# הגדרת המודל
model = ViTForImageClassification.from_pretrained(
    model_name,
    num_labels=len(label_names),
    id2label=id2label,
    label2id=label2id,
    ignore_mismatched_sizes=True
)

metric = evaluate.load("accuracy")

def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    predictions = np.argmax(predictions, axis=1)
    return metric.compute(predictions=predictions, references=labels)

def collate_fn(batch):
    return {
        'pixel_values': torch.stack([x['pixel_values'] for x in batch]),
        'labels': torch.tensor([x['labels'] for x in batch])
    }

training_args = TrainingArguments(
    output_dir="./tomadoc_advanced_ripeness",
    per_device_train_batch_size=32,
    per_device_eval_batch_size=32,
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_strategy="epoch",
    num_train_epochs=5,
    fp16=True,
    learning_rate=2e-5,
    save_total_limit=2,
    remove_unused_columns=False,
    load_best_model_at_end=True,
    metric_for_best_model="accuracy"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    processing_class=processor,
    compute_metrics=compute_metrics,
    data_collator=collate_fn
)

print("מתחיל אימון למודל הבשלות המתקדם...")
trainer.train()

print("מעריך ביצועים על נתוני הבדיקה...")
test_results = trainer.evaluate(test_ds)
accuracy_score = test_results.get('eval_accuracy', 0.0)
print(f"דיוק סופי: {accuracy_score:.4f}")

print("התהליך הושלם בהצלחה.")

מוריד את דטאסט FreshCheck הרשמי מקאגל...
Dataset URL: https://www.kaggle.com/datasets/orofinadedamola/freshcheck-fruit-ripeness-classification-dataset
License(s): CC-BY-SA-4.0
freshcheck-fruit-ripeness-classification-dataset.zip: Skipping, found more recently modified local copy (use --force to force download)
מחלץ את הנתונים...
סורק את התיקיות ומסדר את תמונות העגבנייה...
נמצאו וסודרו בהצלחה 7224 תמונות.
טוען את התמונות למבנה נתונים...


Resolving data files:   0%|          | 0/5779 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/721 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/724 [00:00<?, ?it/s]

Generating train split: 0 examples [00:00, ? examples/s]

...

Training Loss,Validation Loss,Epoch,Accuracy
0.001127,0.007643,5,0.998270


דיוק סופי: 0.9983
מושך את המפתח מתוך קולאב ומבצע התחברות ל-Hugging Face...
שומר את המודל מקומית בנתיב: ./tomadoc_advanced_ripeness_model


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

מעלה את כל הקבצים למאגר הענן: tomadoc-advanced-ripeness-classifier


RepositoryNotFoundError: 404 Client Error. (Request ID: Root=1-6a2b0d86-69ad5d836ec0c8263e89e4c3;94d9536c-8dea-44bb-912c-b16146c19a26)

Repository Not Found for url: https://huggingface.co/api/models/tomadoc-advanced-ripeness-classifier/preupload/main.
Please make sure you specified the correct `repo_id` and `repo_type`.
If you are trying to access a private or gated repo, make sure you are authenticated and your token has the required permissions.
For more details, see https://huggingface.co/docs/huggingface_hub/authentication
Note: Creating a commit assumes that the repo already exists on the Huggingface Hub. Please use `create_repo` if it's not the case.

In [3]:
print("מושך את המפתח מתוך קולאב ומבצע התחברות ל-Hugging Face...")
hf_token = userdata.get('HF_TOKEN')
login(token=hf_token)

local_directory = "./tomadoc_advanced_ripeness_model"
hf_repository_name = "oshriagronov/tomadoc-flash"

print(f"שומר את המודל מקומית בנתיב: {local_directory}")
trainer.save_model(local_directory)
processor.save_pretrained(local_directory)

model_card_content = f"""---
language:
- en
license: mit
tags:
- image-classification
- vit
- agriculture
- tomato-ripeness
- freshcheck
metrics:
- accuracy
pipeline_tag: image-classification
---

# Tomadoc Advanced Tomato Ripeness Classifier

This model is a fine-tuned version of `google/vit-base-patch16-224` designed to identify the exact ripening stage of a tomato. It was trained on the extensive FreshCheck dataset to provide highly accurate agricultural insights for the Tomadoc application.

## Model Description
- **Model Type:** Vision Transformer (ViT)
- **Task:** Image Classification
- **Classes:** Unripe, Ripe, Overripe, Damaged
- **Base Model:** google/vit-base-patch16-224

## Training & Hardware Details
- **Dataset:** FreshCheck Fruit Ripeness Classification Dataset
- **Hardware Used:** NVIDIA T4 GPU
- **Precision:** Mixed Precision (FP16)
- **Evaluation Strategy:** Epoch-based
- **Learning Rate:** 2e-5
- **Epochs:** 5

## Evaluation Results
- **Final Test Accuracy:** {accuracy_score:.4f}
"""

readme_path = os.path.join(local_directory, "README.md")
with open(readme_path, "w", encoding="utf-8") as f:
    f.write(model_card_content)

api = HfApi()
api.create_repo(repo_id=hf_repository_name, exist_ok=True)

print(f"מעלה את כל הקבצים למאגר הענן: {hf_repository_name}")
api.upload_folder(
    folder_path=local_directory,
    repo_id=hf_repository_name,
    commit_message="Upload advanced Tomadoc ripeness model"
)

print("התהליך הושלם בהצלחה.")

מושך את המפתח מתוך קולאב ומבצע התחברות ל-Hugging Face...
שומר את המודל מקומית בנתיב: ./tomadoc_advanced_ripeness_model


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

מעלה את כל הקבצים למאגר הענן: oshriagronov/tomadoc-flash


Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...s_model/model.safetensors:   0%|          |  553kB /  343MB            

  ...s_model/training_args.bin:  48%|####8     | 2.52kB / 5.20kB            

התהליך הושלם בהצלחה.
